# Kraken vs TrOCR benchmark na pelnych stronach EHRI

Porownanie trzech pipeline'ow na pelnych polskich stronach maszynopisu z EHRI:
1. **OpenCV + TrOCR run5** (`trocr-pl-mixed-v3`) - najlepszy model na druk
2. **OpenCV + TrOCR run6** (`trocr-pl-mixed-aug-v1`) - najlepszy model na maszynopis
3. **Kraken end-to-end** (`polish_nfd_9313.mlmodel`) - model EHRI (93,1% accuracy)

Cel: sprawdzic czy Kraken (segmentacja baseline + .mlmodel) rozwiazuje problem
pelnych stron maszynopisu, gdzie OpenCV+TrOCR zawodzi przez slaba segmentacje.

Metryka: CER/WER na poziomie linii (GT z ALTO XML).

In [ ]:
import subprocess, sys, os
from pathlib import Path
CODE_REVISION = '96aa283bc61ce9bce8473a2ca9c7a38007c31557'
BASE_MODEL = 'PiotrSty/trocr-pl-base'
RUN5_MODEL = 'PiotrSty/trocr-pl-mixed-v3'
RUN6_MODEL = 'PiotrSty/trocr-pl-mixed-aug-v1'
EHRI_REVISION = '3003e8614b74a351e7d94aba4f1348368815fb70'
EHRI_DATASET_REPO = 'PiotrSty/ehri-dataset'
KRAKEN_MODEL_FILE = 'models/polish_nfd_9313.mlmodel'
repo = Path('/kaggle/working/OCR_engine')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/PiotrStyla/OCR_engine.git',str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'fetch','origin',CODE_REVISION],check=True)
subprocess.run(['git','-C',str(repo),'checkout','--detach',CODE_REVISION],check=True)
os.chdir(repo)
sys.path.insert(0,str(repo))
subprocess.run([sys.executable,'-m','pip','install','kraken>=5.0','jiwer','pillow','opencv-python-headless'],check=True)
subprocess.run([sys.executable,'-m','pip','uninstall','-y','huggingface_hub'],check=True)
subprocess.run([sys.executable,'-m','pip','install','--no-cache-dir','huggingface_hub==0.36.2'],check=True)
subprocess.run([sys.executable,'-m','pip','install','transformers==4.57.6','peft==0.19.1','accelerate'],check=True)
for _mod in list(sys.modules):
    if _mod == 'huggingface_hub' or _mod.startswith('huggingface_hub.') or _mod == 'transformers' or _mod.startswith('transformers.'):
        del sys.modules[_mod]
from huggingface_hub.utils import HfFolder  # noqa: F401
from transformers import VisionEncoderDecoderModel  # noqa: F401
import kraken
from importlib.metadata import version as _pkg_version
print('IMPORT_OK', 'kraken', _pkg_version('kraken'))

In [ ]:
import torch, json, tarfile, shutil
from pathlib import Path
from huggingface_hub import hf_hub_download, list_repo_files
assert torch.cuda.is_available(), 'GPU required; select Kaggle GPU T4.'
print('GPU:', torch.cuda.get_device_name(0))

ehri_archive = hf_hub_download('PiotrSty/ehri-pl-lines','ehri-pl-lines-v1.tar.gz',repo_type='dataset',revision=EHRI_REVISION)
ehri_root = Path('/kaggle/working/ehri-pl-lines-v1'); ehri_root.mkdir(parents=True, exist_ok=True)
with tarfile.open(ehri_archive,'r:gz') as b: b.extractall(ehri_root, filter='data')

kraken_model_path = hf_hub_download(EHRI_DATASET_REPO, KRAKEN_MODEL_FILE, repo_type='dataset')
print('Kraken model:', kraken_model_path)

test_docs = ['ZIH3010965', 'ZIH3010201']
ehri_pages_dir = Path('/kaggle/working/ehri-pages')
ehri_pages_dir.mkdir(parents=True, exist_ok=True)
all_files = list_repo_files(EHRI_DATASET_REPO, repo_type='dataset')
for doc in test_docs:
    doc_files = [f for f in all_files if doc in f and f.endswith('.tif')]
    for f in doc_files:
        path = hf_hub_download(EHRI_DATASET_REPO, f, repo_type='dataset')
        shutil.copy(path, ehri_pages_dir / Path(f).name)

print('EHRI test lines:', len(list((ehri_root/'test').glob('*.png'))))
print('Full pages:', len(list(ehri_pages_dir.glob('*.tif'))))

In [ ]:
# Benchmark 1: Line-level CER/WER - TrOCR models (run5 and run2-base; run6 not published on HF)
import sys, subprocess
from pathlib import Path
ehri_root = Path('/kaggle/working/ehri-pl-lines-v1')
real_lines = '/kaggle/working/OCR_engine/benchmarks/real-lines-v1/pairs'
for name, model in [('run5', RUN5_MODEL), ('run2-base', BASE_MODEL)]:
    for split, data in [('ehri-test', f'{ehri_root}/test'), ('real-lines-v1', real_lines)]:
        print(f'=== {name} on {split} ===', flush=True)
        subprocess.run([sys.executable,'-m','training.evaluate','--data',data,
            '--model',model,'--device','cuda','--batch-size','16'], check=True)

In [ ]:
# Kraken line-level evaluation on EHRI test + real-lines-v1
import jiwer
from pathlib import Path
from PIL import Image
import kraken.lib.models as models
from kraken.rpred import rpred
from kraken.blla import segment

recognizer = models.load_any(kraken_model_path, device='cuda')

for split_name, test_dir in [
    ('ehri-test', ehri_root/'test'),
    ('real-lines-v1', Path('/kaggle/working/OCR_engine/benchmarks/real-lines-v1/pairs'))
]:
    pairs = sorted(p for p in test_dir.glob('*.png') if p.with_suffix('.txt').exists())
    refs, hyps = [], []
    for img_path in pairs:
        text = img_path.with_suffix('.txt').read_text(encoding='utf-8').strip()
        if not text:
            continue
        img = Image.open(img_path).convert('RGB')
        seg = segment(img)
        if not seg.lines:
            hyps.append('')
            refs.append(text)
            continue
        pred = rpred(recognizer, img, seg)
        pred_text = ' '.join(record.prediction.strip() for record in pred)
        hyps.append(pred_text)
        refs.append(text)
    cer = jiwer.cer(refs, hyps)
    wer = jiwer.wer(refs, hyps)
    print(f'=== Kraken on {split_name} ===')
    print(f'linii:      {len(refs)}')
    print(f'CER:        {cer:.4f}  ({cer*100:.2f}%)')
    print(f'WER:        {wer:.4f}  ({wer*100:.2f}%)')

In [ ]:
# Benchmark 2: Full-page OCR on EHRI test documents
import sys, json
from pathlib import Path
import jiwer

sys.path.insert(0, '/kaggle/working/OCR_engine')
from ocr.config import OcrConfig
from ocr.pipeline import OcrEngine

ehri_pages_dir = Path('/kaggle/working/ehri-pages')
ehri_root = Path('/kaggle/working/ehri-pl-lines-v1')

def load_page_gt(test_dir):
    manifest_path = test_dir.parent / 'manifest.jsonl'
    if not manifest_path.exists():
        return {}
    pages = {}
    for line in manifest_path.read_text(encoding='utf-8').splitlines():
        rec = json.loads(line)
        if rec.get('split') != 'test':
            continue
        page = rec.get('source_page', '')
        if page not in pages:
            pages[page] = []
        pages[page].append(rec['text'])
    return pages

page_gt = load_page_gt(ehri_root)
print('Pages with GT:', len(page_gt))

for backend_name, backend_cfg in [
    ('OpenCV+TrOCR-run5', {'recognizer_backend': 'trocr', 'recognizer_pl': RUN5_MODEL, 'force_language': 'pl'}),
    ('Kraken-e2e', {'recognizer_backend': 'kraken', 'kraken_model': kraken_model_path, 'force_language': 'pl'}),
]:
    print(f'\n=== {backend_name} on full pages ===', flush=True)
    cfg = OcrConfig(**backend_cfg)
    cfg.device = 'cuda'
    engine = OcrEngine(cfg)
    all_refs, all_hyps = [], []
    for page_file in sorted(ehri_pages_dir.glob('*.tif')):
        page_name = page_file.stem
        gt_lines = []
        for gt_page, lines in page_gt.items():
            if page_name in gt_page or gt_page in page_name:
                gt_lines = lines
                break
        if not gt_lines:
            print(f'  {page_name}: no GT, skipping')
            continue
        result = engine.recognize(str(page_file))
        hyp_lines = [ln.text for ln in result.lines if ln.text]
        ref = '\n'.join(gt_lines)
        hyp = '\n'.join(hyp_lines)
        all_refs.append(ref)
        all_hyps.append(hyp)
        print(f'  {page_name}: {len(gt_lines)} GT lines, {len(hyp_lines)} OCR lines')
    if all_refs:
        cer = jiwer.cer(all_refs, all_hyps)
        wer = jiwer.wer(all_refs, all_hyps)
        print(f'  CER: {cer:.4f}  ({cer*100:.2f}%)')
        print(f'  WER: {wer:.4f}  ({wer*100:.2f}%)')
    engine.close()